# MagScan-Mamba: full study on the magnification-shift benchmark (BreakHis, benign vs malignant)

**Proposed model (locked after the two pilots).** MagScan-Mamba reads the patch tokens of a frozen pathology
foundation model with a small selective state-space (Mamba) head:

1. **Frozen Phikon** turns the tile (224 x 224, exactly the benchmark's input) into 14 x 14 patch tokens,
   average-pooled to 7 x 7.
2. **Mamba scan head.** Two layers, each running one Mamba (S6) mixer along four orders of the token grid (row
   by row and column by column, forwards and backwards); then mean pooling and a linear classifier.
3. **Temperature scaling** on the validation set.
4. **Reliability gate.** Mahalanobis distance on the *frozen* Phikon feature flags inputs unlike the training data
   (an unseen magnification, another institution). The pilots showed that trained head features lose this signal.

**What the pilots established** (held-out 400x and 40x only, 3 seeds; pilot notebooks kept as run):
- Reading frozen-FM patch tokens with a small head beats the benchmark's Phikon linear probe in 6 of 6 runs.
- The token mixer does not matter within noise: Mamba 0.890, attention 0.897, gated conv without the SSM 0.880
  (mean macro-F1 over both magnifications). The paper therefore claims the gain for the FM patch tokens, not for
  state-space scanning.
- A three-zoom pyramid helps at 400x and fails at 40x (a tile cannot be zoomed out); it stays as an ablation.
- The design was chosen on 400x and 40x. **100x and 200x were never used, so they are the confirmation.**

**This notebook** (one Kaggle session, GPU T4, about 4 h):

| Part | What | Runs |
|---|---|---|
| LOMO | every variant at held-out 100x and 200x, seeds 13 / 42 / 2025; 400x and 40x restored from the pilots | 48 new + 48 restored |
| CV | 5-fold patient cross-validation (the benchmark's Part C folds, seed 13) at all four held-out magnifications, for the proposed model and its matched ablations | 80 |
| Compute | parameters, FLOPs (fvcore) and latency of backbone + head | - |
| Summary | proposed model against the four benchmark encoders on the same splits (Friedman + Nemenyi over the 12 LOMO blocks), the confirmation on 100x/200x, the ablations, CV, the reliability gate | - |

The confirmation criteria are printed before the results: at 100x and 200x the proposed model's mean macro-F1
must be at least that of every benchmark encoder, and its mean ECE and risk at 80% coverage lower than theirs.

## Run guide

1. **File -> Import Notebook**. Settings: **GPU T4**, **Internet ON**.
2. **Add Input:**
   - `ambarish/breakhis` (required) and `truthisneverlinear/bach-breast-cancer-histology-images`
   - **the pilot 2 output** (`magscan_pilot/runs/...`): its 48 runs at 400x and 40x are restored
   - **all four binary LOMO outputs** (`main_ho40`, `main_ho100`, `main_ho200`, `main_ho400`): the benchmark
     encoders' results on the same splits, and the split check
   - **the Part C cross-validation output(s)** (runs named `cv_owkin_phikon_...`): the CV split check and the
     Phikon CV reference
3. **Save & Run All.** Check in the log: `config hash 964d5fff16571ef4`, `restored 48 finished run(s)`, and
   `identical` in every split check. If a commit stops early, attach this version as an input and run again.
4. Download `magscan_full/magscan_results.zip` (records, tables, figures, compute report; no weights).

In [ ]:
# ---------- Configuration. Split settings and hyperparameters are the pilots' and must not change ----------
# (they enter the config hash; the pilots' runs are restored only while it is unchanged)
HELD_OUT_MAGS = [400, 40, 100, 200]       # 400 and 40 are restored from the pilots; 100 and 200 confirm
CONFIRM_MAGS = [100, 200]                 # never used to choose the design
SEEDS = [13, 42, 2025]                    # the benchmark's seeds
VARIANTS = ['cls_probe', 'cls3_probe', 'noscan', 'attn', 'magscan_single', 'magscan',
            'noscan_single', 'attn_single']
PROPOSED = 'magscan_single'               # MagScan-Mamba: Mamba scan over single-view frozen-Phikon patch tokens
MATCHED_ABLATIONS = ['noscan_single', 'attn_single', 'cls_probe']
RUN_NAME = 'magscan_full'
OUTPUT = f'/kaggle/working/{RUN_NAME}'
BREAKHIS_ROOT = None                      # None = auto-discover under /kaggle/input
BACH_ROOT = None

# ---- 5-fold patient cross-validation: the benchmark's Part C folds ----
RUN_CV = True
CV_K = 5
CV_SEED = 13                              # Part C's CV seed
CV_MAGS = [40, 100, 200, 400]             # Part C cross-validated Phikon at all four
CV_VARIANTS = ['cls_probe', 'noscan_single', 'attn_single', 'magscan_single']
RUN_COMPUTE_REPORT = True

# ---- benchmark protocol (identical to the LOMO training notebooks) ----
MAGNIFICATIONS = [40, 100, 200, 400]      # the training pool; never narrowed
PATIENT_FRACTION = 1.0
IMAGE_SIZE = 224
TASK = 'binary'
NUM_CLASSES = 2
TEMPERATURE_SCALE = True
CHECKPOINT_RULE = 'maximum validation macro-F1; earliest epoch breaks ties; test evaluated once after selection'

# ---- MagScan-Mamba (unchanged from the pilots) ----
FM_NAME = 'owkin/phikon'
ZOOMS = (1, 2, 4)                         # full field, centre 1/2, centre 1/4 (the zoom ablations use all three)
TOKEN_POOL = 2                            # 14 x 14 patch tokens -> 7 x 7 per view
AUG_DRAWS = 2                             # augmented copies of every tile cached for training (plus 1 clean copy)
HEAD_DIM = 256
HEAD_LAYERS = 2
D_STATE = 16
D_CONV = 4
EXPAND = 2
ATTN_HEADS = 4
DROPOUT = 0.1
HEAD_EPOCHS = 30
HEAD_LR = 3e-4
HEAD_WEIGHT_DECAY = 0.05
HEAD_BATCH = 64
PROBE_EPOCHS = 200                        # the benchmark's linear-probe recipe
PROBE_LR = 1e-3
PROBE_WEIGHT_DECAY = 1e-4
PROBE_BATCH = 256
EXTRACT_BATCH = 32
NUM_WORKERS = 4
MAX_RUNTIME_HOURS = 11.5

assert set(HELD_OUT_MAGS) <= set(MAGNIFICATIONS) and set(CV_MAGS) <= set(MAGNIFICATIONS)
assert set(VARIANTS) <= {'cls_probe', 'cls3_probe', 'noscan', 'attn', 'magscan_single', 'magscan',
                         'noscan_single', 'attn_single'}
assert PROPOSED in VARIANTS and set(CV_VARIANTS) <= set(VARIANTS)
_n_lomo = len(HELD_OUT_MAGS) * len(SEEDS) * len(VARIANTS)
_n_cv = len(CV_MAGS) * CV_K * len(CV_VARIANTS) if RUN_CV else 0
print(f'OUTPUT = {OUTPUT}\nLOMO: held-out {HELD_OUT_MAGS} x seeds {SEEDS} x {len(VARIANTS)} variants = {_n_lomo} runs'
      f'\nCV:   {CV_K} folds x {CV_MAGS} x {CV_VARIANTS} = {_n_cv} runs\nproposed: {PROPOSED}')

In [ ]:
# ---------- Environment: packages, the selective-scan kernel, and a check of that kernel ----------
import os, sys, json, math, random, time, hashlib, re, shutil, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone

Path(OUTPUT).mkdir(parents=True, exist_ok=True)


def pip_install_if_missing(module_name, spec=None):
    try:
        __import__(module_name); return True
    except ImportError:
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec or module_name], check=True)
            return True
        except subprocess.CalledProcessError as exc:
            print(f'Install failed for {spec or module_name}:', exc); return False


for _m, _s in [('transformers', 'transformers'), ('einops', 'einops')]:
    pip_install_if_missing(_m, _s)

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, sklearn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             average_precision_score, roc_curve)
from sklearn.covariance import LedoitWolf

assert torch.cuda.is_available(), 'Kaggle GPU accelerator is required (Settings -> Accelerator -> GPU T4).'
DEVICE = torch.device('cuda:0')

# The compiled selective scan of mamba_ssm (the same kernel MedMamba uses). Only selective_scan_fn is needed,
# so causal-conv1d is optional. Without the kernel the pure-torch recurrence below is used: same numbers,
# about 10-20x slower.
SCAN_BACKEND = 'reference'
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn as _cuda_scan
    SCAN_BACKEND = 'cuda'
except Exception:
    # the specs that installed MedMamba's kernel in the benchmark notebooks, in the same order
    for _spec in [['mamba-ssm', 'causal-conv1d'], ['mamba-ssm==2.2.4', 'causal-conv1d==1.5.0.post8'],
                  ['mamba-ssm==2.2.2', 'causal-conv1d==1.4.0']]:
        try:
            print('Installing', _spec)
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation'] + _spec, check=True)
            from mamba_ssm.ops.selective_scan_interface import selective_scan_fn as _cuda_scan
            SCAN_BACKEND = 'cuda'; break
        except Exception as exc:
            print('  failed:', str(exc)[:160])


def selective_scan_reference(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False):
    """Pure-torch selective scan (the recurrence of Gu & Dao, 2024).
    u, delta: (b, d, l); A: (d, n); B, C: (b, g, n, l) with d divisible by g; D: (d,)."""
    dtype_in = u.dtype
    u, delta = u.float(), delta.float()
    if delta_bias is not None:
        delta = delta + delta_bias[..., None].float()
    if delta_softplus:
        delta = F.softplus(delta)
    b, d, l = u.shape
    B = B.float().repeat_interleave(d // B.shape[1], dim=1)              # (b, d, n, l)
    C = C.float().repeat_interleave(d // C.shape[1], dim=1)
    dA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A.float()))       # (b, d, l, n)
    dBu = torch.einsum('bdl,bdnl,bdl->bdln', delta, B, u)                # (b, d, l, n)
    h = u.new_zeros(b, d, A.shape[1])
    ys = []
    for i in range(l):
        h = dA[:, :, i] * h + dBu[:, :, i]
        ys.append(torch.einsum('bdn,bdn->bd', h, C[:, :, :, i]))
    y = torch.stack(ys, dim=2)
    if D is not None:
        y = y + u * D.float()[..., None]
    if z is not None:
        y = y * F.silu(z.float())
    return y.to(dtype_in)


SELECTIVE_SCAN = _cuda_scan if SCAN_BACKEND == 'cuda' else selective_scan_reference
if SCAN_BACKEND == 'cuda':
    # The kernel and the reference must agree before any result is trusted.
    _g = torch.Generator().manual_seed(0)
    _b, _d, _l, _n = 2, 8, 37, 4
    _u, _dt = torch.randn(_b, _d, _l, generator=_g), torch.randn(_b, _d, _l, generator=_g)
    _A = -torch.rand(_d, _n, generator=_g) - 0.1
    _B, _C = torch.randn(_b, 1, _n, _l, generator=_g), torch.randn(_b, 1, _n, _l, generator=_g)
    _D, _bias = torch.randn(_d, generator=_g), torch.randn(_d, generator=_g)
    _ref = selective_scan_reference(_u, _dt, _A, _B, _C, _D, delta_bias=_bias, delta_softplus=True)
    _ker = _cuda_scan(*[t.to(DEVICE) for t in (_u, _dt, _A, _B, _C, _D)], z=None, delta_bias=_bias.to(DEVICE),
                      delta_softplus=True).cpu()
    _diff = float((_ref - _ker).abs().max())
    assert _diff < 1e-3, f'CUDA selective scan disagrees with the reference (max diff {_diff:.2e})'
    print(f'selective scan: CUDA kernel, matches the reference (max diff {_diff:.1e})')
else:
    print('selective scan: pure-torch reference (mamba_ssm unavailable); results are the same, only slower')

env = {'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version, 'torch': torch.__version__,
       'torch_cuda': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0), 'torchvision': torchvision.__version__,
       'sklearn': sklearn.__version__, 'scan_backend': SCAN_BACKEND}
Path(OUTPUT, 'environment.json').write_text(json.dumps(env, indent=2))
Path(OUTPUT, 'config.json').write_text(json.dumps({k: v for k, v in globals().items()
                                                   if k.isupper() and k not in {'OUTPUT', 'DEVICE'}},
                                                  indent=2, default=str))
print(json.dumps(env, indent=2))

In [ ]:
# ---------- BreakHis manifest and the patient-disjoint LOMO split: copied verbatim from the benchmark ----------
# Same file-name pattern, the same row order (sorted paths, no de-duplication) and the same patient_lomo_split,
# so seed s at held-out magnification m gives exactly the benchmark's train / val / test patients. The manifest
# hash is compared with the one stored by any attached benchmark output.
NAME = re.compile(r'^(?P<patient>SOB_(?P<coarse>[BM])_(?P<subtype>[A-Z]+)-\d+-[A-Za-z0-9]+)-(?P<magnification>\d+)-\d+', re.I)
SUBTYPE_TO_ID = {'A': 0, 'F': 1, 'PT': 2, 'TA': 3, 'DC': 4, 'LC': 5, 'MC': 6, 'PC': 7}


def discover_breakhis_root(search_root='/kaggle/input'):
    candidates = []
    if Path(search_root).exists():
        candidates = [top for top in sorted(Path(search_root).iterdir()) if top.is_dir()] + [Path(search_root)]
    best, best_count = None, 0
    for candidate in candidates:
        try:
            count = sum(1 for _ in Path(candidate).rglob('SOB_*.png'))
        except (PermissionError, OSError):
            continue
        if count > best_count:
            best, best_count = candidate, count
    if best is None or best_count == 0:
        raise RuntimeError('Could not auto-discover BreakHis root. Attach ambarish/breakhis or set BREAKHIS_ROOT.')
    print('Auto-discovered BREAKHIS_ROOT =', best, ' (', best_count, 'SOB PNG files)')
    return str(best)


def build_manifest(root):
    rows = []
    for path in sorted(Path(root).rglob('SOB_*.png')):
        match = NAME.match(path.name)
        if not match:
            continue
        item = match.groupdict()
        subtype = item['subtype'].upper()
        if subtype not in SUBTYPE_TO_ID:
            continue
        rows.append({'path': str(path), 'patient_id': item['patient'], 'magnification': int(item['magnification']),
                     'coarse_label': int(item['coarse'].upper() == 'M'), 'subtype': subtype,
                     'subtype_label': SUBTYPE_TO_ID[subtype]})
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError('No matching images found under ' + str(root))
    return frame


def patient_lomo_split(frame, held_out_mag, fraction=1.0, seed=2025,
                       val_ratio=0.15, test_ratio=0.20, val_on_held_out=False):
    """Patient-disjoint LOMO split with per-class stratification (the benchmark's function, unchanged)."""
    frame = frame.copy()
    patient_class = frame.groupby('patient_id').coarse_label.first().reset_index()
    rng = np.random.default_rng(seed)
    train_patients, val_patients, test_patients = set(), set(), set()
    for cls in sorted(patient_class.coarse_label.unique()):
        group = patient_class.loc[patient_class.coarse_label == cls, 'patient_id'].to_numpy()
        perm = rng.permutation(group)
        n = len(perm)
        n_test = max(1, int(round(test_ratio * n)))
        n_val = max(1, int(round(val_ratio * n)))
        assert n_test + n_val < n, 'Class ' + str(cls) + ' has only ' + str(n) + ' patients; val+test ratios too high.'
        test_patients.update(perm[:n_test].tolist())
        val_patients.update(perm[n_test:n_test + n_val].tolist())
        remain = perm[n_test + n_val:].tolist()
        n_keep = max(1, int(round(fraction * len(remain))))
        train_patients.update(remain[:n_keep])
    train_mags = [m for m in MAGNIFICATIONS if m != held_out_mag]
    frame['split'] = 'drop'
    frame.loc[frame.patient_id.isin(train_patients) & frame.magnification.isin(train_mags), 'split'] = 'train'
    if val_on_held_out:
        frame.loc[frame.patient_id.isin(val_patients) & (frame.magnification == held_out_mag), 'split'] = 'val'
    else:
        frame.loc[frame.patient_id.isin(val_patients) & frame.magnification.isin(train_mags), 'split'] = 'val'
    frame.loc[frame.patient_id.isin(test_patients) & (frame.magnification == held_out_mag), 'split'] = 'test'
    subset = frame.loc[frame.split != 'drop'].copy()
    leak = subset.groupby('patient_id').split.nunique().gt(1).any()
    assert not leak, 'Patient leakage detected.'
    for name in ('train', 'val', 'test'):
        rows = subset[subset.split == name]
        if len(rows) == 0:
            raise RuntimeError('Split ' + name + ' empty (held_out=' + str(held_out_mag) + ', seed=' + str(seed) + ').')
        if rows.coarse_label.nunique() < 2:
            raise RuntimeError('Split ' + name + ' has only one class -- stratification failed.')
    return subset


if BREAKHIS_ROOT is None or not list(Path(BREAKHIS_ROOT).rglob('SOB_*.png')):
    BREAKHIS_ROOT = discover_breakhis_root()
MANIFEST = build_manifest(BREAKHIS_ROOT)
MANIFEST.to_csv(Path(OUTPUT, 'breakhis_manifest.csv'), index=False)
MANIFEST_HASH = hashlib.sha256(MANIFEST.to_csv(index=False).encode()).hexdigest()      # the benchmark's hash (with paths)
# The same rows without the mount point: identifies the data across sessions wherever the dataset is mounted.
MANIFEST_KEY = hashlib.sha256(MANIFEST.assign(path=MANIFEST.path.map(lambda p: Path(p).name))
                              .to_csv(index=False).encode()).hexdigest()
print('total images:', len(MANIFEST), '  patients:', MANIFEST.patient_id.nunique(), '  manifest SHA-256:', MANIFEST_HASH)

# ---- attached benchmark outputs: their manifest hash and runs (for the split check and the reference rows) ----
BENCHMARK_DIRS = []
if Path('/kaggle/input').exists():
    BENCHMARK_DIRS = sorted({p.parent.parent.parent for p in Path('/kaggle/input').rglob('record.json')
                             if p.parent.parent.name == 'runs'
                             and p.parent.name.startswith(('scratch_', 'fm_owkin_phikon_', 'cv_owkin_phikon_'))})
_hashes = {str(d): Path(d, 'breakhis_manifest.sha256').read_text().strip()
           for d in BENCHMARK_DIRS if Path(d, 'breakhis_manifest.sha256').exists()}
for d, h in _hashes.items():
    # The benchmark notebooks hash their manifest CSV the same way, but their path column holds their own
    # mount point, so a differing hash is expected when the dataset is mounted elsewhere; the test-label
    # check in the training cell is the decisive one.
    print(f'  benchmark output {d}: manifest hash {"identical" if h == MANIFEST_HASH else "differs (mount path)"}')
print(f'{len(BENCHMARK_DIRS)} benchmark output folder(s) attached' + ('' if BENCHMARK_DIRS else
      ' - the pilot runs anyway; attach them to see the reference rows and the split check'))


# ---- 5-fold patient cross-validation split: copied verbatim from the benchmark's Part C notebook ----
def patient_kfold_lomo_split(frame, held_out_mag, k, fold_idx, seed, val_ratio=0.15, fraction=1.0):
    """Part C's split: test = this fold's patients at the held-out magnification (1/k), val = val_ratio of all
    patients of the class (rotating window), train = the rest; class balance in every split."""
    assert 0 <= fold_idx < k, f'fold_idx {fold_idx} out of range for k={k}'
    frame = frame.copy()
    patient_class = frame.groupby('patient_id').coarse_label.first().reset_index()
    rng = np.random.default_rng(seed)
    fold_test, fold_val, fold_train = {}, {}, {}
    for cls in sorted(patient_class.coarse_label.unique()):
        group = patient_class.loc[patient_class.coarse_label == cls, 'patient_id'].to_numpy()
        perm = rng.permutation(group)
        buckets = np.array_split(perm, k)
        test = list(buckets[fold_idx])
        remaining = np.concatenate([buckets[i] for i in range(k) if i != fold_idx])
        n_val = max(1, int(round(val_ratio * len(perm))))
        assert n_val < len(remaining), f'class {cls}: val size {n_val} >= remaining {len(remaining)}'
        span = max(1, len(remaining) - n_val)
        val_start = (fold_idx * n_val) % span
        val = list(remaining[val_start:val_start + n_val])
        train_pool = list(remaining[:val_start]) + list(remaining[val_start + n_val:])
        n_keep = max(1, int(round(fraction * len(train_pool))))
        train = train_pool[:n_keep]
        fold_test.setdefault(cls, []).extend(test)
        fold_val.setdefault(cls, []).extend(val)
        fold_train.setdefault(cls, []).extend(train)
    test_p, val_p, train_p = (set(sum(d.values(), [])) for d in (fold_test, fold_val, fold_train))
    train_mags = [m for m in MAGNIFICATIONS if m != held_out_mag]
    frame['split'] = 'drop'
    frame.loc[frame.patient_id.isin(train_p) & frame.magnification.isin(train_mags), 'split'] = 'train'
    frame.loc[frame.patient_id.isin(val_p) & frame.magnification.isin(train_mags), 'split'] = 'val'
    frame.loc[frame.patient_id.isin(test_p) & (frame.magnification == held_out_mag), 'split'] = 'test'
    subset = frame.loc[frame.split != 'drop'].copy()
    assert not subset.groupby('patient_id').split.nunique().gt(1).any(), f'Patient leakage in fold {fold_idx}.'
    for name in ('train', 'val', 'test'):
        rows = subset[subset.split == name]
        if len(rows) == 0:
            raise RuntimeError(f'Fold {fold_idx} split {name} empty (held_out={held_out_mag}).')
        if rows.coarse_label.nunique() < 2:
            raise RuntimeError(f'Fold {fold_idx} split {name} has only one class.')
    return subset


if RUN_CV:
    _tp = [set(patient_kfold_lomo_split(MANIFEST, 400, CV_K, f, CV_SEED).query("split == 'test'").patient_id)
           for f in range(CV_K)]
    assert all(not (_tp[a] & _tp[b]) for a in range(CV_K) for b in range(a + 1, CV_K)), 'CV test folds overlap'
    print(f'CV: {CV_K} folds (seed {CV_SEED}); test patients per fold at 400x: {[len(t) for t in _tp]}, '
          f'{len(set().union(*_tp))} of {MANIFEST.patient_id.nunique()} patients covered')

In [ ]:
# ---------- Metrics: the benchmark's calibration / selective-prediction code and section 2g's OOD scores ----------
_trapz = getattr(np, 'trapezoid', None) or np.trapz


def expected_calibration_error(probs, labels, n_bins=15):
    confidences = probs.max(axis=1); predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lower, upper in zip(bins[:-1], bins[1:]):
        in_bin = (confidences > lower) & (confidences <= upper)
        if in_bin.sum() == 0: continue
        ece += in_bin.mean() * abs(accuracies[in_bin].mean() - confidences[in_bin].mean())
    return float(ece)


def brier_multiclass(probs, labels):
    return float(((probs - np.eye(probs.shape[1])[labels]) ** 2).sum(axis=1).mean())


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__(); self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        return logits / torch.exp(self.log_temp)


def fit_temperature(val_logits, val_labels, max_iter=200):
    scaler = TemperatureScaler().to(DEVICE)
    logits = torch.as_tensor(val_logits, device=DEVICE, dtype=torch.float32)
    labels = torch.as_tensor(val_labels, device=DEVICE, dtype=torch.long)
    optim = torch.optim.LBFGS([scaler.log_temp], lr=0.1, max_iter=max_iter)
    loss_fn = nn.CrossEntropyLoss()

    def closure():
        optim.zero_grad(); loss = loss_fn(scaler(logits), labels); loss.backward(); return loss
    optim.step(closure)
    return float(torch.exp(scaler.log_temp).item())


def risk_coverage_curve(probs, labels):
    confidences = probs.max(axis=1); predictions = probs.argmax(axis=1)
    order = np.argsort(-confidences)
    errors = (predictions != labels).astype(float)[order]
    coverage = np.arange(1, len(order) + 1) / len(order)
    risk = np.cumsum(errors) / np.arange(1, len(order) + 1)
    return coverage, risk, float(_trapz(risk, coverage))


def compute_all_metrics(logits, labels, temperature=1.0):
    scaled = logits / temperature
    exp = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    probs = exp / exp.sum(axis=1, keepdims=True)
    predictions = probs.argmax(axis=1)
    metrics = {'accuracy': accuracy_score(labels, predictions),
               'balanced_accuracy': balanced_accuracy_score(labels, predictions),
               'macro_f1': f1_score(labels, predictions, average='macro'),
               'ece': expected_calibration_error(probs, labels), 'brier': brier_multiclass(probs, labels)}
    try:
        metrics['auc'] = roc_auc_score(labels, probs[:, 1])
    except ValueError:
        metrics['auc'] = float('nan')
    coverage, risk, rcauc = risk_coverage_curve(probs, labels)
    metrics['risk_coverage_auc'] = rcauc
    for target in (0.80, 0.90, 0.95):
        idx = int(target * len(coverage)) - 1
        metrics['risk_at_coverage_' + str(int(100 * target))] = float(risk[idx]) if idx >= 0 else float('nan')
    # E-AURC (Geifman et al., ICLR 2019): AURC minus its lowest value at this error rate
    r = min(1.0 - metrics['accuracy'], 1 - 1e-12)
    metrics['e_aurc'] = rcauc - (r + (1 - r) * np.log(1 - r))
    metrics['temperature'] = float(temperature)
    return metrics, probs


# ---- OOD scores, as in the analysis notebook's section 2g (higher = more out-of-distribution) ----
def energy_from_logits(logits):
    z = np.asarray(logits, dtype=np.float64); m = z.max(axis=1, keepdims=True)
    return -(m[:, 0] + np.log(np.exp(z - m).sum(axis=1)))


def neg_msp(logits):
    z = np.asarray(logits, dtype=np.float64); z = z - z.max(axis=1, keepdims=True)
    p = np.exp(z); p /= p.sum(axis=1, keepdims=True)
    return -p.max(axis=1)


def fit_mahalanobis(train_feats, train_labels):
    """Class means + one Ledoit-Wolf tied covariance on class-centred training features."""
    X = np.asarray(train_feats, dtype=np.float64); y = np.asarray(train_labels).astype(int)
    classes = np.unique(y)
    means = np.stack([X[y == c].mean(axis=0) for c in classes])
    precision = LedoitWolf(assume_centered=True).fit(X - means[np.searchsorted(classes, y)]).precision_
    return means, precision


def mahalanobis_score(feats, means, precision):
    X = np.asarray(feats, dtype=np.float64)
    return np.min(np.stack([np.einsum('nd,nd->n', (X - mu) @ precision, X - mu) for mu in means], axis=1), axis=1)


def knn_score(feats, train_feats, k=10, chunk=2048):
    def _n(a):
        a = np.asarray(a, dtype=np.float64); return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)
    bank, q = _n(train_feats), _n(feats)
    k = int(min(k, len(bank))); out = np.empty(len(q))
    for s in range(0, len(q), chunk):
        sims = q[s:s + chunk] @ bank.T
        kth = -np.partition(-sims, k - 1, axis=1)[:, k - 1]
        out[s:s + chunk] = np.sqrt(np.clip(2.0 - 2.0 * kth, 0.0, None))
    return out


def ood_auroc(id_scores, ood_scores):
    y = np.r_[np.zeros(len(id_scores)), np.ones(len(ood_scores))]
    return float(roc_auc_score(y, np.r_[id_scores, ood_scores]))


print('metrics ready')

In [ ]:
# ---------- Frozen Phikon tokens for every tile, zoom view and augmentation draw (extracted once) ----------
# Draw 0 is the clean tile (validation, test, the Mahalanobis reference); draws 1..AUG_DRAWS apply the
# benchmark's training augmentation (flips, rotation up to 15 degrees, colour jitter) to the full tile before it
# is cut into zoom views, so the three views of a draw show the same augmented tissue. Each draw is seeded by
# (draw, tile), so it is identical in every session. Stored per tile: (zoom, 1 CLS + 49 pooled patch tokens, 768)
# in float16; Phikon itself runs in float32, as in the benchmark.
from transformers import AutoModel

_NORMALIZE = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(),
                                 transforms.Normalize((.485, .456, .406), (.229, .224, .225))])
_TRAIN_AUG = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                                 transforms.RandomRotation(15), transforms.ColorJitter(.1, .1, .05)])


def zoom_views(img):
    """(Z, 3, 224, 224): centre crops of 1/z of the tile's width and height, each resized to 224 x 224.
    Zoom 1 is the whole tile, i.e. exactly the benchmark's input."""
    w, h = img.size
    views = []
    for z in ZOOMS:
        cw, ch = max(1, round(w / z)), max(1, round(h / z))
        left, top = (w - cw) // 2, (h - ch) // 2
        views.append(_NORMALIZE(img.crop((left, top, left + cw, top + ch))))
    return torch.stack(views)


class ZoomDataset(Dataset):
    def __init__(self, paths, draw):
        self.paths, self.draw = list(paths), int(draw)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        if self.draw > 0:
            torch.manual_seed(1_000_003 * self.draw + i)
            img = _TRAIN_AUG(img)
        return zoom_views(img), i


FM = AutoModel.from_pretrained(FM_NAME, token=os.environ.get('HF_TOKEN'), trust_remote_code=True).eval().to(DEVICE)
for _p in FM.parameters():
    _p.requires_grad_(False)


@torch.inference_mode()
def extract_tokens(paths, draw):
    loader = DataLoader(ZoomDataset(paths, draw), batch_size=EXTRACT_BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    out = None
    for views, idx in loader:
        b, z = views.shape[:2]
        h = FM(pixel_values=views.flatten(0, 1).to(DEVICE, non_blocking=True)).last_hidden_state   # (b*z, 1+g*g, D)
        g = int(round((h.shape[1] - 1) ** 0.5))
        assert g * g == h.shape[1] - 1, f'unexpected token count {h.shape[1]}'
        grid = h[:, 1:].reshape(b * z, g, g, -1).permute(0, 3, 1, 2)                           # (b*z, D, g, g)
        pooled = F.avg_pool2d(grid, TOKEN_POOL).flatten(2).transpose(1, 2)                     # (b*z, (g/p)^2, D)
        tok = torch.cat([h[:, :1], pooled], dim=1).reshape(b, z, -1, h.shape[-1])
        if out is None:
            out = torch.empty((len(loader.dataset),) + tuple(tok.shape[1:]), dtype=torch.float16)
        out[idx] = tok.half().cpu()
    return out


t0 = time.perf_counter()
TOKENS = {d: extract_tokens(MANIFEST.path.tolist(), d) for d in range(AUG_DRAWS + 1)}
LABELS = MANIFEST.coarse_label.to_numpy().astype(int)
N_TOKENS_PER_VIEW = TOKENS[0].shape[2] - 1
FM_DIM = TOKENS[0].shape[-1]
print(f'BreakHis tokens: {len(TOKENS)} draws x {tuple(TOKENS[0].shape)} float16 '
      f'({sum(t.numel() for t in TOKENS.values()) * 2 / 2**30:.1f} GiB) in {(time.perf_counter() - t0) / 60:.1f} min')

# Zoom 1 of draw 0 must be the benchmark's own Phikon input: compare its CLS with a direct forward pass.
with torch.inference_mode():
    _x = torch.stack([_NORMALIZE(Image.open(p).convert('RGB')) for p in MANIFEST.path[:4]]).to(DEVICE)
    _cls = FM(pixel_values=_x).last_hidden_state[:, 0].float().cpu()
_d = float((_cls - TOKENS[0][:4, 0, 0].float()).abs().max() / _cls.abs().max())
assert _d < 5e-3, f'zoom-1 CLS differs from the benchmark input (relative {_d:.1e})'
print(f'zoom-1 CLS = the benchmark\'s Phikon feature (relative diff {_d:.1e}, float16 storage)')


# ---- BACH (another institution), clean views only ----
def discover_bach_root(search_root='/kaggle/input'):
    best, best_count = None, 0
    if Path(search_root).exists():
        for root, dirs, _files in os.walk(search_root):
            if {'Normal', 'Benign', 'InSitu', 'Invasive'}.issubset(set(dirs)):
                count = sum(1 for _ in Path(root).rglob('*.tif')) + sum(1 for _ in Path(root).rglob('*.png'))
                if count > best_count:
                    best, best_count = Path(root), count
    if best is not None:
        print(f'BACH_ROOT = {best} ({best_count} image files)')
    return None if best is None else str(best)


BACH_TOKENS = None
if BACH_ROOT is None:
    BACH_ROOT = discover_bach_root()
if BACH_ROOT is not None:
    _bach_paths = [str(p) for cls in ('Normal', 'Benign', 'InSitu', 'Invasive')
                   for sfx in ('*.tif', '*.png', '*.jpg') for p in sorted(Path(BACH_ROOT, cls).glob(sfx))]
    BACH_TOKENS = extract_tokens(_bach_paths, 0)
    print(f'BACH tokens: {tuple(BACH_TOKENS.shape)}')
else:
    print('BACH not attached: the cross-institution score is skipped')

In [ ]:
# ---------- MagScan-Mamba and its ablations ----------
# Input per tile: (Z, 1 + 49, 768) Phikon tokens. The token heads use the 49 pooled patch tokens of each view;
# the probes use the CLS tokens. All token heads share everything except the token mixer:
#   LayerNorm -> Linear(768, D) + scale embedding + 2-D position embedding
#   -> HEAD_LAYERS x [ x + mixer(LN(x)) ; x + MLP(LN(x)) ] -> LN -> mean over tokens -> dropout -> Linear(D, 2)
# mixer = 'mamba' : one selective-SSM mixer (Mamba block) run along four orders of the (zoom, row, col) grid -
#                   coarse->fine row-major, its reverse, coarse->fine column-major, its reverse - mapped back
#                   to the original order and averaged. Weights are shared across the four orders.
#         'conv'  : the same block with the SSM removed: gated causal depthwise conv only (MambaOut, CVPR 2025).
#         'attn'  : multi-head self-attention over all tokens.
class MambaMixer(nn.Module):
    """Mamba (S6) block of Gu & Dao (2024): in_proj -> causal depthwise conv -> SiLU -> selective scan,
    gated by SiLU(z) -> out_proj. use_ssm=False drops the selective scan and keeps the gated conv."""
    def __init__(self, d_model, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND, use_ssm=True):
        super().__init__()
        self.d_inner, self.d_state, self.use_ssm = expand * d_model, d_state, use_ssm
        self.dt_rank = math.ceil(d_model / 16)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, d_conv, groups=self.d_inner, padding=d_conv - 1)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        if use_ssm:
            self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
            self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
            nn.init.uniform_(self.dt_proj.weight, -self.dt_rank ** -0.5, self.dt_rank ** -0.5)
            dt = torch.exp(torch.rand(self.d_inner) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)).clamp(min=1e-4)
            with torch.no_grad():
                self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))          # softplus^-1(dt), as in Mamba
            self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)))
            self.D = nn.Parameter(torch.ones(self.d_inner))

    def forward(self, x):                                                          # x: (B, L, d_model)
        L = x.shape[1]
        u, z = self.in_proj(x).chunk(2, dim=-1)                                    # (B, L, d_inner) each
        u = F.silu(self.conv1d(u.transpose(1, 2))[..., :L])                        # (B, d_inner, L), causal
        if self.use_ssm:
            dt, Bm, Cm = torch.split(self.x_proj(u.transpose(1, 2)), [self.dt_rank, self.d_state, self.d_state], dim=-1)
            dt = (dt @ self.dt_proj.weight.t()).transpose(1, 2)                   # (B, d_inner, L); bias in the scan
            u = SELECTIVE_SCAN(u.float().contiguous(), dt.float().contiguous(), -torch.exp(self.A_log.float()),
                               Bm.transpose(1, 2).unsqueeze(1).float().contiguous(),   # (B, 1, N, L)
                               Cm.transpose(1, 2).unsqueeze(1).float().contiguous(),
                               self.D.float(), z=None, delta_bias=self.dt_proj.bias.float(), delta_softplus=True)
        return self.out_proj(u.transpose(1, 2) * F.silu(z))


def scan_orders(n_zoom, grid):
    """Four index orders over tokens stored as (zoom, row, col); each maps back with argsort."""
    idx = torch.arange(n_zoom * grid * grid).reshape(n_zoom, grid, grid)
    row_major = idx.flatten()
    col_major = idx.transpose(1, 2).flatten()
    orders = [row_major, row_major.flip(0), col_major, col_major.flip(0)]
    return [(o, torch.argsort(o)) for o in orders]


class ScanMixer(nn.Module):
    def __init__(self, d_model, n_zoom, grid, use_ssm=True):
        super().__init__()
        self.core = MambaMixer(d_model, use_ssm=use_ssm)
        for k, (o, inv) in enumerate(scan_orders(n_zoom, grid)):
            self.register_buffer(f'order{k}', o, persistent=False)
            self.register_buffer(f'inverse{k}', inv, persistent=False)

    def forward(self, x):
        out = 0
        for k in range(4):
            o, inv = getattr(self, f'order{k}'), getattr(self, f'inverse{k}')
            out = out + self.core(x[:, o])[:, inv]
        return out / 4


class AttnMixer(nn.Module):
    def __init__(self, d_model):
        super().__init__(); self.attn = nn.MultiheadAttention(d_model, ATTN_HEADS, dropout=DROPOUT, batch_first=True)

    def forward(self, x):
        return self.attn(x, x, x, need_weights=False)[0]


class TokenHead(nn.Module):
    def __init__(self, mixer, n_zoom, grid, in_dim, d=HEAD_DIM, layers=HEAD_LAYERS, num_classes=NUM_CLASSES):
        super().__init__()
        self.n_zoom, self.grid = n_zoom, grid
        self.inp = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, d))
        self.scale_emb = nn.Parameter(torch.zeros(1, n_zoom, 1, d))
        self.pos_emb = nn.Parameter(torch.zeros(1, 1, grid * grid, d))
        nn.init.trunc_normal_(self.scale_emb, std=0.02); nn.init.trunc_normal_(self.pos_emb, std=0.02)
        make = {'mamba': lambda: ScanMixer(d, n_zoom, grid, True), 'conv': lambda: ScanMixer(d, n_zoom, grid, False),
                'attn': lambda: AttnMixer(d)}[mixer]
        self.blocks = nn.ModuleList([nn.ModuleDict({'n1': nn.LayerNorm(d), 'mix': make(), 'n2': nn.LayerNorm(d),
                                                    'mlp': nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(DROPOUT),
                                                                         nn.Linear(2 * d, d))})
                                     for _ in range(layers)])
        self.norm, self.drop, self.fc = nn.LayerNorm(d), nn.Dropout(DROPOUT), nn.Linear(d, num_classes)

    def features(self, tokens):                            # tokens: (B, Z, 1 + grid^2, in_dim) -> (B, d)
        x = self.inp(tokens[:, :, 1:].float()) + self.scale_emb + self.pos_emb     # (B, Z, grid^2, d)
        x = x.flatten(1, 2)                                                         # (B, Z * grid^2, d): (zoom, row, col)
        for blk in self.blocks:
            x = x + blk['mix'](blk['n1'](x))
            x = x + blk['mlp'](blk['n2'](x))
        return self.norm(x).mean(dim=1)

    def forward(self, tokens):
        return self.fc(self.drop(self.features(tokens)))


class ClsProbe(nn.Module):
    """Linear probe on the CLS token of each used view (one view = the benchmark's Phikon probe)."""
    def __init__(self, n_zoom, in_dim, num_classes=NUM_CLASSES):
        super().__init__(); self.fc = nn.Linear(n_zoom * in_dim, num_classes)

    def features(self, tokens):
        return tokens[:, :, 0].float().flatten(1)

    def forward(self, tokens):
        return self.fc(self.features(tokens))


VARIANT_SPEC = {   # variant: (kind, zoom indices into ZOOMS)
    'cls_probe': ('probe', [0]), 'cls3_probe': ('probe', list(range(len(ZOOMS)))),
    'noscan': ('conv', list(range(len(ZOOMS)))), 'attn': ('attn', list(range(len(ZOOMS)))),
    'magscan_single': ('mamba', [0]), 'magscan': ('mamba', list(range(len(ZOOMS)))),
    'noscan_single': ('conv', [0]), 'attn_single': ('attn', [0])}      # pilot 2: matched to magscan_single
GRID = int(round(N_TOKENS_PER_VIEW ** 0.5))


def build_variant(variant):
    kind, zooms = VARIANT_SPEC[variant]
    if kind == 'probe':
        return ClsProbe(len(zooms), FM_DIM)
    return TokenHead(kind, len(zooms), GRID, FM_DIM)


# ---- self-checks: orders invert, shapes, and the parameter budget of every head ----
for o, inv in scan_orders(3, GRID):
    assert torch.equal(o[inv], torch.arange(len(o))), 'scan order does not invert'
_rows = []
for v in VARIANTS:
    m = build_variant(v).to(DEVICE)
    zs = VARIANT_SPEC[v][1]
    with torch.no_grad():
        out = m(torch.randn(2, len(zs), 1 + GRID * GRID, FM_DIM, device=DEVICE))
    assert out.shape == (2, NUM_CLASSES)
    _rows.append({'variant': v, 'trainable_params': sum(p.numel() for p in m.parameters() if p.requires_grad),
                  'tokens': len(zs) * GRID * GRID if VARIANT_SPEC[v][0] != 'probe' else len(zs)})
    del m
PARAMS = pd.DataFrame(_rows).set_index('variant')
print(PARAMS.to_string())

In [ ]:
# ---------- Train and evaluate one variant on one split: LOMO (held-out, seed) or a CV fold ----------
# Splits: the benchmark's patient_lomo_split, or Part C's patient_kfold_lomo_split for a CV fold. Training tiles use a random augmented draw each epoch; validation,
# test and the Mahalanobis reference use the clean draw. Checkpoint: best validation macro-F1 (earliest epoch on
# ties), then temperature scaling on the validation logits, then one test evaluation - the benchmark's rule.
# Probes follow the benchmark's probe recipe (AdamW 1e-3, 200 epochs, batch 256); token heads use AdamW 3e-4
# with cosine decay for 30 epochs, batch 64. The loss is unweighted cross-entropy, as in the benchmark.
import copy

CONFIG_KEYS = ['MANIFEST_KEY', 'ZOOMS', 'TOKEN_POOL', 'AUG_DRAWS', 'HEAD_DIM', 'HEAD_LAYERS', 'D_STATE', 'D_CONV',
               'EXPAND', 'ATTN_HEADS', 'DROPOUT', 'HEAD_EPOCHS', 'HEAD_LR', 'HEAD_WEIGHT_DECAY', 'HEAD_BATCH',
               'PROBE_EPOCHS', 'PROBE_LR', 'PROBE_WEIGHT_DECAY', 'PROBE_BATCH', 'FM_NAME', 'PATIENT_FRACTION']
CONFIG_HASH = hashlib.sha256(json.dumps({k: globals()[k] for k in CONFIG_KEYS}, default=str,
                                        sort_keys=True).encode()).hexdigest()[:16]
OOD_PAIRS = {'test_vs_val': ('val', 'test'), 'bach_vs_test': ('test', 'bach')}


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def run_id_for(variant, ho, seed, fold=None):
    if fold is None:                                   # the pilots' ids, so their runs are restored
        return f'magscan_{variant}_ho{ho}_frac{PATIENT_FRACTION}_seed{seed}'
    return f'magscan_cv_{variant}_ho{ho}_frac{PATIENT_FRACTION}_seed{seed}_fold{fold}'


def split_indices(ho, seed, fold=None):
    sub = (patient_lomo_split(MANIFEST, ho, fraction=PATIENT_FRACTION, seed=seed) if fold is None else
           patient_kfold_lomo_split(MANIFEST, ho, CV_K, fold, seed, fraction=PATIENT_FRACTION))
    return {s: np.array(sub.index[sub.split == s]) for s in ('train', 'val', 'test')}      # manifest rows, in order


def benchmark_split_check(ho, seed, test_rows, fold=None):
    """Compare the test labels with the stored ones of the benchmark's Phikon probe run (same order):
    the LOMO run, or Part C's CV run of the same fold."""
    rid = (f'fm_owkin_phikon_ho{ho}_frac{PATIENT_FRACTION}_seed{seed}' if fold is None else
           f'cv_owkin_phikon_ho{ho}_frac{PATIENT_FRACTION}_seed{seed}_fold{fold}')
    for d in BENCHMARK_DIRS:
        f = Path(d, 'runs', rid, 'test_outputs.npz')
        if f.exists():
            stored = np.load(f)['labels']
            return 'identical' if np.array_equal(stored, LABELS[test_rows]) else 'DIFFERS'
    return 'benchmark run not attached'


@torch.no_grad()
def predict(model, X, batch=256):
    model.eval()
    logits, feats = [], []
    for s in range(0, len(X), batch):
        xb = X[s:s + batch].to(DEVICE)
        f = model.features(xb)
        feats.append(f.float().cpu()); logits.append(model.fc(f).float().cpu())
    return torch.cat(logits).numpy(), torch.cat(feats).numpy()


def run_variant(variant, ho, seed, fold=None):
    seed_everything(seed)
    kind, zooms = VARIANT_SPEC[variant]
    idx = split_indices(ho, seed, fold)
    zsel = torch.as_tensor(zooms)
    token_slice = slice(0, 1) if kind == 'probe' else slice(None)        # probes need only the CLS token

    def gather(draw, rows):
        return TOKENS[draw][torch.as_tensor(rows)][:, zsel][:, :, token_slice]

    draws = list(range(1, AUG_DRAWS + 1)) or [0]
    X_train = torch.stack([gather(d, idx['train']) for d in draws]).to(DEVICE)   # (draws, n, Z, T, D) float16
    X = {s: gather(0, idx[s]) for s in ('train', 'val', 'test')}                   # clean, on the CPU
    y = {s: LABELS[idx[s]] for s in ('train', 'val', 'test')}
    y_train_dev = torch.as_tensor(y['train'], device=DEVICE)

    model = build_variant(variant).to(DEVICE)
    probe = kind == 'probe'
    epochs, lr, wd, bs = ((PROBE_EPOCHS, PROBE_LR, PROBE_WEIGHT_DECAY, PROBE_BATCH) if probe else
                          (HEAD_EPOCHS, HEAD_LR, HEAD_WEIGHT_DECAY, HEAD_BATCH))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    n = len(y['train']); steps = math.ceil(n / bs)
    sched = None if probe else torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * steps)
    loss_fn = nn.CrossEntropyLoss()
    gen = torch.Generator().manual_seed(seed)
    best, best_state, best_epoch, log = -1.0, None, 0, []
    torch.cuda.reset_peak_memory_stats(); start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train(); running = 0.0
        order = torch.randperm(n, generator=gen)
        for s in range(0, n, bs):
            b = order[s:s + bs]
            d = torch.randint(0, len(draws), (len(b),), generator=gen)              # one draw per tile per epoch
            xb = X_train[d.to(DEVICE), b.to(DEVICE)]
            loss = loss_fn(model(xb), y_train_dev[b.to(DEVICE)])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            if sched is not None:
                sched.step()
            running += float(loss.detach()) * len(b)
        vl, _ = predict(model, X['val'])
        vm, _ = compute_all_metrics(vl, y['val'])
        log.append({'epoch': epoch, 'train_loss': running / n, **{f'val_{k}': v for k, v in vm.items()}})
        if vm['macro_f1'] > best:                                                   # strict: earliest epoch wins ties
            best, best_epoch = vm['macro_f1'], epoch
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    del X_train
    val_logits, val_feats = predict(model, X['val'])
    temperature = fit_temperature(val_logits, y['val']) if TEMPERATURE_SCALE else 1.0
    test_logits, test_feats = predict(model, X['test'])
    tm, test_probs = compute_all_metrics(test_logits, y['test'], temperature)
    wall = time.perf_counter() - start

    # ---- reliability: does the score flag the unseen magnification (test vs val) and another institution? ----
    _, train_feats = predict(model, X['train'])
    logits = {'val': val_logits, 'test': test_logits}
    feats = {'val': val_feats, 'test': test_feats}
    if BACH_TOKENS is not None:
        logits['bach'], feats['bach'] = predict(model, BACH_TOKENS[:, zsel][:, :, token_slice])
    means, prec = fit_mahalanobis(train_feats, y['train'])
    scores = {'maha': {s: mahalanobis_score(f, means, prec) for s, f in feats.items()},
              'knn': {s: knn_score(f, train_feats) for s, f in feats.items()},
              'msp': {s: neg_msp(l) for s, l in logits.items()},
              'energy': {s: energy_from_logits(l) for s, l in logits.items()}}
    ood = {f'{score}_{pair}_auroc': ood_auroc(sc[a], sc[b])
           for score, sc in scores.items() for pair, (a, b) in OOD_PAIRS.items() if a in sc and b in sc}

    run_id = run_id_for(variant, ho, seed, fold)
    run_dir = Path(OUTPUT, 'runs', run_id); run_dir.mkdir(parents=True, exist_ok=True)
    record = {'run_id': run_id, 'branch': 'magscan_lomo' if fold is None else 'magscan_cv',
              'model': f'magscan:{variant}', 'variant': variant,
              'held_out_mag': ho, 'patient_fraction': PATIENT_FRACTION, 'seed': seed, 'fold_idx': fold,
              'n_train': int(n), 'n_val': len(y['val']), 'n_test': len(y['test']), 'best_epoch': best_epoch,
              'trainable_params': int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
              'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20, 'wallclock_s': wall,
              'split_check': benchmark_split_check(ho, seed, idx['test'], fold), 'config_hash': CONFIG_HASH,
              'scan_backend': SCAN_BACKEND, **{f'test_{k}': v for k, v in tm.items()}, **ood}
    pd.DataFrame(log).to_csv(run_dir / 'epochs.csv', index=False)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=y['test'], probs=test_probs,
                        manifest_index=idx['test'])
    torch.save({'state_dict': {k: v.cpu() for k, v in best_state.items()}, 'variant': variant,
                'temperature': temperature}, run_dir / 'head.pt')
    (run_dir / 'record.json').write_text(json.dumps(record, indent=2, default=str))       # written last = finished
    del model; torch.cuda.empty_cache()
    return record


print('config hash', CONFIG_HASH)

In [ ]:
# ---------- Run every LOMO (held-out, seed, variant) and CV (held-out, fold, variant); resumable ----------
# A run is finished when runs/<run_id>/record.json exists. Finished runs of attached earlier outputs (the pilots,
# or an earlier version of this notebook) are restored first, but only if they were produced with the same
# configuration (config hash): same data, same hyperparameters.
if Path('/kaggle/input').exists():
    _restored, _other_hash = 0, set()
    for rec in Path('/kaggle/input').rglob('record.json'):
        if rec.parent.parent.name != 'runs' or not rec.parent.name.startswith('magscan_'):
            continue
        try:
            r = json.loads(rec.read_text())
        except Exception:
            continue
        dst = Path(OUTPUT, 'runs', rec.parent.name)
        if r.get('config_hash') == CONFIG_HASH and not (dst / 'record.json').exists():
            shutil.copytree(rec.parent, dst, dirs_exist_ok=True); _restored += 1
        elif r.get('config_hash') != CONFIG_HASH:
            _other_hash.add(r.get('config_hash'))
    print(f'restored {_restored} finished run(s) from attached earlier outputs')
    if _other_hash:
        print(f'  WARNING: attached runs with another configuration were NOT restored (their hash '
              f'{sorted(map(str, _other_hash))}, this session {CONFIG_HASH}); they will be re-run')

# ---- split checks first: stop before any GPU time is spent if a split differs from the benchmark's ----
_checks = {('lomo', ho, seed): benchmark_split_check(ho, seed, split_indices(ho, seed)['test'])
           for ho in HELD_OUT_MAGS for seed in SEEDS}
if RUN_CV:
    _checks.update({(f'cv fold {f}', ho, CV_SEED): benchmark_split_check(ho, CV_SEED, split_indices(ho, CV_SEED, f)['test'], f)
                    for ho in CV_MAGS for f in range(CV_K)})
_status = pd.Series(list(_checks.values())).value_counts().to_dict()
print(f'split checks ({len(_checks)}): {_status}')
for (kind, ho, seed), status in _checks.items():
    if status != 'identical':
        print(f'  {kind:<10} held-out {ho}x seed {seed}: {status}')
assert 'DIFFERS' not in _checks.values(), 'a test split differs from the benchmark: stop and check the manifest'

try:
    import psutil
    KERNEL_START = psutil.Process(os.getpid()).create_time()
except Exception:
    KERNEL_START = time.time()
deadline = KERNEL_START + (MAX_RUNTIME_HOURS - 0.3) * 3600

planned = [(ho, seed, v, None) for ho in HELD_OUT_MAGS for seed in SEEDS for v in VARIANTS]
if RUN_CV:
    planned += [(ho, CV_SEED, v, f) for ho in CV_MAGS for f in range(CV_K) for v in CV_VARIANTS]
records, longest = [], {}
for i, (ho, seed, v, fold) in enumerate(planned, 1):
    rid = run_id_for(v, ho, seed, fold)
    f = Path(OUTPUT, 'runs', rid, 'record.json')
    if f.exists():
        records.append(json.loads(f.read_text())); continue
    need = 1.2 * longest.get(v, 300.0)
    if time.time() + need > deadline:
        print(f'[{i}/{len(planned)}] not started (time budget): {rid}'); continue
    t0 = time.perf_counter()
    try:
        rec = run_variant(v, ho, seed, fold)
        records.append(rec)
        longest[v] = max(longest.get(v, 0.0), time.perf_counter() - t0)
        print(f'[{i}/{len(planned)}] {rid:<48} F1 {rec["test_macro_f1"]:.3f}  ECE {rec["test_ece"]:.3f}  '
              f'risk@80 {rec["test_risk_at_coverage_80"]:.3f}  ({(time.perf_counter() - t0) / 60:.1f} min, '
              f'split {rec["split_check"]})')
    except Exception as exc:
        import traceback; traceback.print_exc()
        print(f'[{i}/{len(planned)}] FAILED {rid}: {repr(exc)[:300]}')

RUNS = pd.DataFrame(records)
if len(RUNS):
    RUNS['branch'] = RUNS.branch.replace({'magscan_pilot': 'magscan_lomo'})      # the pilots' LOMO runs
    if 'fold_idx' not in RUNS.columns:
        RUNS['fold_idx'] = np.nan
    # a restored run keeps the split status of the session that made it; this session checked every split
    # up front, so its result is the current one
    def _current_check(r):
        f = r.get('fold_idx')
        key = ('lomo', r.held_out_mag, r.seed) if f is None or f != f else (f'cv fold {int(f)}', r.held_out_mag, r.seed)
        return _checks.get(key, r.split_check)
    RUNS['split_check'] = RUNS.apply(_current_check, axis=1)
    RUNS.to_csv(Path(OUTPUT, 'magscan_runs.csv'), index=False)
    bad = RUNS[RUNS.split_check == 'DIFFERS']
    assert bad.empty, f'test labels differ from the benchmark for {bad.run_id.tolist()}'
LOMO = RUNS[RUNS.branch == 'magscan_lomo'] if len(RUNS) else RUNS
CV = RUNS[RUNS.branch == 'magscan_cv'] if len(RUNS) else RUNS
print(f'{len(RUNS)} of {len(planned)} runs finished ({len(LOMO)} LOMO, {len(CV)} CV)')

In [ ]:
# ---------- Compute: parameters, FLOPs and throughput of the frozen backbone + each head ----------
# FLOPs are counted with fvcore (one fused multiply-add = one FLOP, as in the benchmark's compute report) at
# batch 1, 224 x 224. fvcore cannot see the custom CUDA selective scan, so each head is counted with the
# pure-torch reference scan, which does the same arithmetic. A variant with Z views runs the backbone Z times.
# Throughput: tiles per second on this GPU at batch 32 (backbone + head, float32, as trained).
COMPUTE = None
if RUN_COMPUTE_REPORT:
    try:
        from fvcore.nn import FlopCountAnalysis
    except ImportError:
        pip_install_if_missing('fvcore')
    try:
        from fvcore.nn import FlopCountAnalysis

        def _gflops(model, *inputs):
            fa = FlopCountAnalysis(model, inputs)
            fa.unsupported_ops_warnings(False); fa.uncalled_modules_warnings(False)
            return fa.total() / 1e9
    except Exception as exc:
        print('fvcore unavailable, FLOPs skipped:', repr(exc)[:150])
        _gflops = None

    class _Backbone(nn.Module):
        def __init__(self, fm):
            super().__init__(); self.fm = fm

        def forward(self, x):
            return self.fm(pixel_values=x).last_hidden_state

    _bb = _Backbone(FM).eval()
    fm_params = sum(p.numel() for p in FM.parameters())
    fm_gflops = float('nan')
    if _gflops is not None:
        try:
            with torch.no_grad():
                fm_gflops = _gflops(_bb, torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))
        except Exception as exc:
            print('FLOP count failed for the backbone:', repr(exc)[:150])

    def _throughput(model, n_views, batch=32, reps=10):
        x = torch.randn(batch * n_views, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        with torch.no_grad():
            for _ in range(3):                                  # warm-up
                h = _bb(x)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(reps):
                h = _bb(x)
                g = int(round((h.shape[1] - 1) ** 0.5))
                grid = h[:, 1:].reshape(h.shape[0], g, g, -1).permute(0, 3, 1, 2)
                pooled = F.avg_pool2d(grid, TOKEN_POOL).flatten(2).transpose(1, 2)
                tok = torch.cat([h[:, :1], pooled], dim=1).reshape(batch, n_views, -1, h.shape[-1])
                model(tok)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
        return batch * reps / (time.perf_counter() - t0)

    rows = []
    for v in VARIANTS:
        kind, zooms = VARIANT_SPEC[v]
        m = build_variant(v).to(DEVICE).eval()
        trainable = sum(p.numel() for p in m.parameters())
        head_gflops = float('nan')
        if _gflops is not None:
            _saved_scan = SELECTIVE_SCAN
            SELECTIVE_SCAN = selective_scan_reference           # fvcore traces the pure-torch recurrence
            try:
                with torch.no_grad():
                    head_gflops = _gflops(m, torch.randn(1, len(zooms), 1 + GRID * GRID, FM_DIM, device=DEVICE))
            except Exception as exc:
                print(f'FLOP count failed for the {v} head:', repr(exc)[:150])
            finally:
                SELECTIVE_SCAN = _saved_scan
        try:
            tput = _throughput(m, len(zooms))
        except Exception as exc:
            tput = float('nan'); print(f'throughput failed for {v}:', repr(exc)[:150])
        rows.append({'model': f'magscan:{v}', 'variant': v, 'views': len(zooms), 'trainable_params': trainable,
                     'frozen_params': fm_params, 'parameters': fm_params + trainable,
                     'backbone_gflops': fm_gflops * len(zooms), 'head_gflops': head_gflops,
                     'flops_g_at_224': fm_gflops * len(zooms) + head_gflops, 'tiles_per_s': tput})
        del m
    COMPUTE = pd.DataFrame(rows)
    COMPUTE.to_csv(Path(OUTPUT, 'magscan_compute_report.csv'), index=False)
    print(f'frozen Phikon: {fm_params / 1e6:.1f} M parameters, {fm_gflops:.2f} GFLOPs per view')
    print(COMPUTE[['variant', 'views', 'trainable_params', 'head_gflops', 'flops_g_at_224', 'tiles_per_s']]
          .round(3).to_string(index=False))

In [ ]:
# ---------- Results: MagScan-Mamba against the benchmark, the confirmation, ablations, CV, gate, compute ----------
import matplotlib.pyplot as plt
import zipfile
from scipy.stats import friedmanchisquare

CONFIRMATION_RULE = f"""Confirmation criteria (fixed before these results; {CONFIRM_MAGS} were never used for the design):
  (1) accuracy     at each of {CONFIRM_MAGS}, {PROPOSED}'s mean macro-F1 is at least that of every benchmark encoder
  (2) reliability  at each of them, its mean ECE and its mean risk at 80% coverage are below every benchmark encoder's"""
print(CONFIRMATION_RULE)

NAME = {PROPOSED: 'MagScan-Mamba', 'convnext_tiny': 'ConvNeXt-T', 'swin_tiny': 'Swin-T', 'medmamba': 'MedMamba',
        'owkin/phikon': 'Phikon (probe)'}
BENCH_ORDER = ['MagScan-Mamba', 'ConvNeXt-T', 'Swin-T', 'MedMamba', 'Phikon (probe)']
MAIN_METRICS = [('test_macro_f1', 'Macro-F1', True), ('test_balanced_accuracy', 'Bal. acc.', True),
                ('test_ece', 'ECE', False), ('test_brier', 'Brier', False),
                ('test_risk_at_coverage_80', 'Risk@80', False), ('test_risk_coverage_auc', 'AURC', False),
                ('test_e_aurc', 'E-AURC', False)]
TABLES = Path(OUTPUT, 'tables'); TABLES.mkdir(exist_ok=True)
pd.set_option('display.width', 250)


def _e_aurc(frame):
    err = (1 - frame.test_accuracy).clip(0, 1 - 1e-12)
    return frame.test_risk_coverage_auc - (err + (1 - err) * np.log(1 - err))


def _pm(frame, col):
    return frame[col].mean(), frame[col].std()


# ---- 1. the benchmark encoders' records on the same splits (attached LOMO and Part C outputs) ----
_bench, _bench_cv = [], []
for d in BENCHMARK_DIRS:
    for rec in Path(d, 'runs').glob('*/record.json'):
        r = json.loads(rec.read_text())
        if float(r.get('patient_fraction', 1.0)) != PATIENT_FRACTION:
            continue
        if r.get('branch') in ('from_scratch', 'fm_linear_probe') and r.get('seed') in SEEDS:
            _bench.append(r)
        elif r.get('branch') == 'cv' and r.get('model') == FM_NAME:
            _bench_cv.append(r)
BENCH = pd.DataFrame(_bench).drop_duplicates('run_id') if _bench else pd.DataFrame()
BENCH_CV = pd.DataFrame(_bench_cv).drop_duplicates('run_id') if _bench_cv else pd.DataFrame()
if len(BENCH):
    BENCH['test_e_aurc'] = _e_aurc(BENCH)
    BENCH['method'] = BENCH.model.map(NAME)
print(f'\nbenchmark records found: {len(BENCH)} LOMO runs '
      f'({BENCH.groupby("method").size().to_dict() if len(BENCH) else "none - attach the four LOMO outputs"}), '
      f'{len(BENCH_CV)} Phikon CV runs')

MAIN = LOMO[LOMO.variant == PROPOSED].assign(method='MagScan-Mamba') if len(LOMO) else LOMO
COMBO = pd.concat([MAIN, BENCH], ignore_index=True) if len(BENCH) else MAIN.copy()
_cols = [c for c, _, _ in MAIN_METRICS if c in COMBO.columns]

# ---- 2. per held-out magnification: MagScan-Mamba next to the four benchmark encoders ----
if len(COMBO):
    per_mag = COMBO.groupby(['held_out_mag', 'method'])[_cols].agg(['mean', 'std'])
    per_mag.to_csv(TABLES / 'main_per_magnification.csv')
    for ho in HELD_OUT_MAGS:
        if ho not in per_mag.index.get_level_values(0):
            continue
        t = per_mag.loc[ho].reindex([m for m in BENCH_ORDER if m in per_mag.loc[ho].index])
        show = pd.DataFrame({lab: t[(c, 'mean')].map('{:.3f}'.format) + ' ± ' + t[(c, 'std')].fillna(0).map('{:.3f}'.format)
                             for c, lab, _ in MAIN_METRICS if c in _cols}, index=t.index)
        tag = '   <- confirmation (not used for the design)' if ho in CONFIRM_MAGS else '   (design chosen here)'
        print(f'\n=== held-out {ho}x: mean ± SD over seeds{tag} ===')
        print(show.to_string())

    # ---- 3. over all held-out magnifications: means, and Friedman + Nemenyi over the (held-out, seed) blocks ----
    overall = COMBO.groupby('method')[_cols].mean().reindex([m for m in BENCH_ORDER if m in set(COMBO.method)])
    overall.to_csv(TABLES / 'main_overall.csv')
    print(f'\n=== mean over every held-out magnification and seed ===')
    print(overall.round(3).to_string())
    NEMENYI_Q05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850}
    _rank_rows = []
    for c, lab, higher in MAIN_METRICS:
        if c not in _cols:
            continue
        block = COMBO.pivot_table(index=['held_out_mag', 'seed'], columns='method', values=c)
        block = block[[m for m in BENCH_ORDER if m in block.columns]].dropna()
        k, n = block.shape[1], len(block)
        if k < 3 or n < 2:
            continue
        avg = block.rank(axis=1, ascending=not higher).mean()
        p = friedmanchisquare(*[block[m] for m in block.columns]).pvalue
        cd = NEMENYI_Q05[k] * np.sqrt(k * (k + 1) / (6.0 * n))
        others = [m for m in block.columns if m != 'MagScan-Mamba']
        sep = [m for m in others if 'MagScan-Mamba' in block.columns and abs(avg['MagScan-Mamba'] - avg[m]) > cd]
        _rank_rows.append({'metric': lab, 'blocks': n, 'friedman_p': p, 'critical_difference': cd,
                           **{f'rank {m}': avg[m] for m in block.columns},
                           'MagScan-Mamba separated from': ', '.join(sep) or 'none'})
    RANKS = pd.DataFrame(_rank_rows)
    if len(RANKS):
        RANKS.to_csv(TABLES / 'main_friedman_nemenyi.csv', index=False)
        print('\n=== mean rank over the (held-out, seed) blocks, 1 = best; Friedman p; Nemenyi CD (alpha 0.05) ===')
        print(RANKS.round(3).to_string(index=False))

    # ---- 4. the confirmation criteria at the untouched magnifications ----
    def _mean(method, ho, c):
        s = COMBO[(COMBO.method == method) & (COMBO.held_out_mag == ho)][c]
        return s.mean() if len(s) else np.nan
    conf_rows = []
    for ho in CONFIRM_MAGS:
        bench_methods = [m for m in BENCH_ORDER[1:] if not np.isnan(_mean(m, ho, 'test_macro_f1'))]
        if np.isnan(_mean('MagScan-Mamba', ho, 'test_macro_f1')) or not bench_methods:
            conf_rows.append({'held_out_mag': ho, 'status': 'not evaluable (runs or benchmark records missing)'})
            continue
        f1_ok = all(_mean('MagScan-Mamba', ho, 'test_macro_f1') >= _mean(m, ho, 'test_macro_f1') for m in bench_methods)
        rel_ok = all(_mean('MagScan-Mamba', ho, c) < _mean(m, ho, c) for m in bench_methods
                     for c in ('test_ece', 'test_risk_at_coverage_80'))
        best = max(bench_methods, key=lambda m: _mean(m, ho, 'test_macro_f1'))
        conf_rows.append({'held_out_mag': ho, '(1) accuracy': 'PASS' if f1_ok else 'fail',
                          '(2) reliability': 'PASS' if rel_ok else 'fail',
                          'MagScan F1': _mean('MagScan-Mamba', ho, 'test_macro_f1'),
                          'best benchmark': best, 'its F1': _mean(best, ho, 'test_macro_f1'),
                          'benchmarks compared': len(bench_methods)})
    CONFIRM = pd.DataFrame(conf_rows)
    CONFIRM.to_csv(TABLES / 'confirmation.csv', index=False)
    print('\n=== CONFIRMATION at the magnifications not used for the design ===')
    print(CONFIRM.round(3).to_string(index=False))

# ---- 5. ablations: every variant at every held-out magnification, and paired differences ----
if len(LOMO):
    abl = LOMO.groupby(['held_out_mag', 'variant'])[['test_macro_f1', 'test_ece', 'test_risk_at_coverage_80']].mean()
    abl.to_csv(TABLES / 'ablations_per_magnification.csv')
    print('\n=== ablations: macro-F1 per held-out magnification (mean over seeds) ===')
    print(abl.test_macro_f1.unstack('held_out_mag').reindex(VARIANTS).round(3).to_string())
    _p = LOMO[LOMO.variant == PROPOSED].set_index(['held_out_mag', 'seed'])
    drows = []
    for v in VARIANTS:
        if v == PROPOSED:
            continue
        o = LOMO[LOMO.variant == v].set_index(['held_out_mag', 'seed'])
        common = _p.index.intersection(o.index)
        if not len(common):
            continue
        for c, lab, higher in [('test_macro_f1', 'Macro-F1', True), ('test_ece', 'ECE', False),
                               ('test_risk_at_coverage_80', 'Risk@80', False)]:
            d = _p.loc[common, c] - o.loc[common, c]
            drows.append({'versus': v, 'metric': lab, 'runs': len(common), 'delta_mean': d.mean(), 'delta_sd': d.std(),
                          'MagScan better in': int(((d > 0) if higher else (d < 0)).sum())})
    ABL_DELTAS = pd.DataFrame(drows)
    ABL_DELTAS.to_csv(TABLES / 'ablation_paired_deltas.csv', index=False)
    print(f'\n=== {PROPOSED} minus each variant, paired by (held-out, seed) over all magnifications ===')
    print(ABL_DELTAS.round(3).to_string(index=False))

# ---- 6. cross-validation: 5 patient folds at each held-out magnification ----
if len(CV):
    cvs = CV.groupby(['held_out_mag', 'variant'])[['test_macro_f1', 'test_ece', 'test_risk_at_coverage_80']].agg(['mean', 'std'])
    cvs.to_csv(TABLES / 'cv_per_magnification.csv')
    print('\n=== 5-fold patient CV: macro-F1 mean ± SD over folds ===')
    _t = CV.groupby(['variant', 'held_out_mag']).test_macro_f1.agg(['mean', 'std'])
    print((_t['mean'].map('{:.3f}'.format) + ' ± ' + _t['std'].map('{:.3f}'.format)).unstack('held_out_mag')
          .reindex([v for v in CV_VARIANTS if v in set(CV.variant)]).to_string())
    if len(BENCH_CV):
        print('\n  Phikon linear probe, the benchmark\'s own Part C CV (same folds):')
        print(BENCH_CV.groupby('held_out_mag').test_macro_f1.agg(['mean', 'std']).round(3).T.to_string())
    _cv_main = CV[CV.variant == PROPOSED].groupby('held_out_mag').test_macro_f1.mean()
    _lomo_main = LOMO[LOMO.variant == PROPOSED].groupby('held_out_mag').test_macro_f1.mean()
    print('\n  MagScan-Mamba, LOMO seeds vs CV folds (macro-F1):')
    print(pd.DataFrame({'LOMO (3 seeds)': _lomo_main, 'CV (5 folds)': _cv_main}).round(3).T.to_string())

# ---- 7. the reliability gate: Mahalanobis (and kNN) on the frozen Phikon feature of the full view ----
_gate_rows = []
_cls = TOKENS[0][:, 0, 0].float().numpy()                           # frozen CLS of zoom 1 (the full tile)
_bach_cls = BACH_TOKENS[:, 0, 0].float().numpy() if BACH_TOKENS is not None else None
for ho in HELD_OUT_MAGS:
    for seed in SEEDS:
        idx = split_indices(ho, seed)
        means, prec = fit_mahalanobis(_cls[idx['train']], LABELS[idx['train']])
        sc = {s: mahalanobis_score(_cls[idx[s]], means, prec) for s in ('val', 'test')}
        row = {'held_out_mag': ho, 'seed': seed, 'maha_test_vs_val': ood_auroc(sc['val'], sc['test']),
               'knn_test_vs_val': ood_auroc(knn_score(_cls[idx['val']], _cls[idx['train']]),
                                            knn_score(_cls[idx['test']], _cls[idx['train']]))}
        if _bach_cls is not None:
            row['maha_bach_vs_test'] = ood_auroc(sc['test'], mahalanobis_score(_bach_cls, means, prec))
            row['knn_bach_vs_test'] = ood_auroc(knn_score(_cls[idx['test']], _cls[idx['train']]),
                                                knn_score(_bach_cls, _cls[idx['train']]))
        _gate_rows.append(row)
GATE = pd.DataFrame(_gate_rows)
GATE.to_csv(TABLES / 'reliability_gate.csv', index=False)
print('\n=== reliability gate on the frozen feature: AUROC, mean over seeds ===')
print(GATE.groupby('held_out_mag').mean(numeric_only=True).drop(columns='seed').round(3).to_string())

# ---- 8. compute ----
if COMPUTE is not None:
    print('\n=== compute (backbone + head) ===')
    print(COMPUTE[['variant', 'views', 'trainable_params', 'flops_g_at_224', 'tiles_per_s']].round(2).to_string(index=False))

# ---- figures ----
if len(COMBO):
    _colors = {'MagScan-Mamba': '#7B2D8E', 'ConvNeXt-T': '#0072B2', 'Swin-T': '#009E73', 'MedMamba': '#CC79A7',
               'Phikon (probe)': '#D55E00'}
    _markers = {'MagScan-Mamba': '*', 'ConvNeXt-T': 'o', 'Swin-T': 's', 'MedMamba': '^', 'Phikon (probe)': 'D'}
    mags = sorted(COMBO.held_out_mag.unique())
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    for ax, (c, lab, higher) in zip(axes, [m for m in MAIN_METRICS if m[0] in ('test_macro_f1', 'test_ece', 'test_risk_at_coverage_80')]):
        for j, m in enumerate([m for m in BENCH_ORDER if m in set(COMBO.method)]):
            g = COMBO[COMBO.method == m].groupby('held_out_mag')[c].agg(['mean', 'std']).reindex(mags)
            xs = np.arange(len(mags)) + (j - 2) * 0.08
            ax.errorbar(xs, g['mean'], yerr=g['std'].fillna(0), color=_colors.get(m, '#666'), marker=_markers.get(m, 'o'),
                        markersize=13 if m == 'MagScan-Mamba' else 7, linewidth=2.2 if m == 'MagScan-Mamba' else 1.2,
                        capsize=3, label=m, markeredgecolor='black', markeredgewidth=0.5)
        for k, ho in enumerate(mags):
            if ho in CONFIRM_MAGS:
                ax.axvspan(k - 0.4, k + 0.4, color='#FFF3C4', zorder=0)
        ax.set_xticks(range(len(mags))); ax.set_xticklabels([f'{int(h)}x' for h in mags])
        ax.set_xlabel('Held-out magnification (shaded = confirmation)')
        ax.set_title(f"{lab} ({'higher' if higher else 'lower'} is better)", fontweight='bold')
        ax.grid(alpha=0.3)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=len(l), frameon=False, bbox_to_anchor=(0.5, -0.04))
    fig.suptitle('MagScan-Mamba against the benchmark encoders (same patient splits; mean ± SD over seeds)',
                 fontweight='bold')
    fig.tight_layout(rect=(0, 0.05, 1, 0.95))
    fig.savefig(Path(OUTPUT, 'magscan_vs_benchmark.png'), dpi=200, bbox_inches='tight')
    fig.savefig(Path(OUTPUT, 'magscan_vs_benchmark.pdf'), bbox_inches='tight')
    plt.show()

if len(LOMO):
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
    _vc = {'cls_probe': '#D55E00', 'cls3_probe': '#E69F00', 'noscan': '#BBBBBB', 'attn': '#9CC9EA',
           'magscan': '#E8A6CF', 'noscan_single': '#666666', 'attn_single': '#0072B2', PROPOSED: '#7B2D8E'}
    mags = sorted(LOMO.held_out_mag.unique())
    width = 0.8 / len(VARIANTS)
    for ax, (c, lab) in zip(axes, [('test_macro_f1', 'Macro-F1 (higher is better)'), ('test_ece', 'ECE (lower is better)')]):
        for j, v in enumerate(VARIANTS):
            g = LOMO[LOMO.variant == v].groupby('held_out_mag')[c].mean().reindex(mags)
            ax.bar(np.arange(len(mags)) + (j - len(VARIANTS) / 2 + 0.5) * width, g, width, color=_vc.get(v, '#999'),
                   edgecolor='black' if v == PROPOSED else 'none', linewidth=1.2, label=v)
        ax.set_xticks(range(len(mags))); ax.set_xticklabels([f'{int(h)}x' for h in mags])
        ax.set_title(lab, fontweight='bold'); ax.grid(axis='y', alpha=0.3)
        vals = LOMO[c].dropna()
        ax.set_ylim(max(0, vals.min() - 0.05), vals.max() + 0.03)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.1))
    fig.suptitle(f'Ablations (mean over seeds): the proposed model is {PROPOSED} (black outline)', fontweight='bold')
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))
    fig.savefig(Path(OUTPUT, 'magscan_ablations.png'), dpi=200, bbox_inches='tight')
    fig.savefig(Path(OUTPUT, 'magscan_ablations.pdf'), bbox_inches='tight')
    plt.show()

# ---- one ZIP with everything needed for the analysis and the paper (no weights) ----
with zipfile.ZipFile(Path(OUTPUT, 'magscan_results.zip'), 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in Path(OUTPUT).rglob('*'):
        if p.is_file() and p.suffix in ('.csv', '.json', '.png', '.pdf', '.npz') and p.name != 'magscan_results.zip':
            zf.write(p, p.relative_to(OUTPUT))
print(f'\nDownload {OUTPUT}/magscan_results.zip (records, test outputs, tables, figures; no weights).')